<a href="https://colab.research.google.com/github/takuonakashima/ai-security-workshop/blob/main/Poison_Training_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

# ==========================================
# 1. 工場のセンサーデータ（正常な学習データ）の生成
# ==========================================
# X: [振動センサーの値, 温度センサーの値], y: 0(正常) または 1(異常)
X, y = make_classification(n_samples=100, n_features=2, n_informative=2,
                           n_redundant=0, n_clusters_per_class=1,
                           class_sep=1.5, random_state=42)

# ==========================================
# 2. 攻撃前：クリーンなデータでAIを学習させる
# ==========================================
clf_clean = LogisticRegression()
clf_clean.fit(X, y)

# ==========================================
# 3. 攻撃の実行（ポイズニングデータの混入）
# ==========================================
# 攻撃者が「異常な振動と温度」なのに「正常(0)」という嘘のラベルをつけたデータを5個だけ混入させる
X_poison = np.array([[-2.0, 3.0], [-2.5, 2.5], [-3.0, 2.0], [-2.0, 2.0], [-1.5, 3.5]])
y_poison = np.array([0, 0, 0, 0, 0]) # 本当は異常エリアなのに「正常(0)」と嘘を教える

# 元のクリーンなデータに、毒入りデータを混ぜる
X_poisoned = np.vstack((X, X_poison))
y_poisoned = np.concatenate((y, y_poison))

# ==========================================
# 4. 攻撃後：汚染されたデータでAIを再学習させる
# ==========================================
clf_poisoned = LogisticRegression()
clf_poisoned.fit(X_poisoned, y_poisoned)

# ==========================================
# 5. 結果のグラフ化（視覚化）
# ==========================================
def plot_decision_boundary(clf, X, y, ax, title):
    # グラフの描画範囲を設定
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02), np.arange(y_min, y_max, 0.02))

    # AIによる判定エリアの描画
    Z = clf.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.coolwarm)

    # 実際のデータポイントのプロット
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap=plt.cm.coolwarm)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Vibration (振動)', fontsize=12)
    ax.set_ylabel('Temperature (温度)', fontsize=12)

# グラフを左右に2つ並べて描画
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 左：攻撃前の安全なAI
plot_decision_boundary(clf_clean, X, y, ax1, '1. Before Attack (Clean AI)')

# 右：ポイズニング攻撃を受けたAI（毒データを星印で強調）
plot_decision_boundary(clf_poisoned, X_poisoned, y_poisoned, ax2, '2. After Poisoning Attack (Compromised AI)')
ax2.scatter(X_poison[:, 0], X_poison[:, 1], c='yellow', marker='*', s=200, edgecolors='black', label='Poison Data')
ax2.legend()

plt.tight_layout()
plt.show()